# Семинар 7
# Оптимизации при обучении нейросетей. Профилирование DL кода. Mixed-precision обучение. Хранение данных и оптимизации загрузки.

Кудин Степан, Senior MLE, Ozon Tech

4 апреля 2026

## Вспоминаем как обучаются нейросети

In [1]:
import multiprocessing
import time
from itertools import chain

import numpy as np
import torch
from torch.utils.data import DataLoader
import tqdm
import datasets
from datasets import DatasetDict
from transformers import AutoConfig, AutoModelForCausalLM, AutoTokenizer, default_data_collator

In [2]:
# Настройки
MODEL_NAME = "openai-community/gpt2"
DATASET_NAME = "tatsu-lab/alpaca"
DATASET_SUBSET = None
NUM_EPOCHS = 2
LR = 3e-5
BATCH_SIZE = 2
SEQ_LENGTH = 1024
RANDOM_SEED = 42

In [3]:
# Получаем параметры
device = torch.device("cuda:0")
dtype = torch.float32

# Фиксируем random seed для стабильности результатов.
torch.manual_seed(RANDOM_SEED)

In [4]:
# Инициализируем модель
config = AutoConfig.from_pretrained(MODEL_NAME, use_cache=False)
model = AutoModelForCausalLM.from_config(config, dtype=dtype)
model.to(device)
model.device

device(type='cuda', index=0)

In [5]:
# Source: https://github.com/LambdaLabsML/distributed-training-guide/blob/main/01-single-gpu/train_llm.py
def load_and_preprocess_data(config, model_name, dataset_name, dataset_subset, seq_length):
    """
    Function created using code found in
    https://github.com/huggingface/transformers/blob/v4.45.1/examples/pytorch/language-modeling/run_clm_no_trainer.py
    """
    tokenizer = AutoTokenizer.from_pretrained(model_name)

    data = datasets.load_dataset(dataset_name)

    column_names = data["train"].column_names
    text_column_name = "text" if "text" in column_names else column_names[0]

    def tokenize_function(examples):
        return tokenizer(examples[text_column_name])

    tokenized_datasets = data.map(
        tokenize_function,
        batched=True,
        remove_columns=column_names,
        num_proc=multiprocessing.cpu_count(),
        load_from_cache_file=True,
        desc="Running tokenizer on dataset",
    )

    seq_length = seq_length or tokenizer.model_max_length
    if seq_length > config.max_position_embeddings:
        seq_length = min(1024, config.max_position_embeddings)

    # Main data processing function that will concatenate all texts from our dataset and generate chunks of block_size.
    def group_texts(examples):
        # Concatenate all texts.
        concatenated_examples = {k: list(chain(*examples[k])) for k in examples.keys()}
        total_length = len(concatenated_examples[list(examples.keys())[0]])
        # We drop the small remainder, and if the total_length < block_size  we exclude this batch and return an empty dict.
        # We could add padding if the model supported it instead of this drop, you can customize this part to your needs.
        if total_length > seq_length:
            total_length = (total_length // seq_length) * seq_length
        # Split by chunks of max_len.
        result = {
            k: [t[i : i + seq_length] for i in range(0, total_length, seq_length)]
            for k, t in concatenated_examples.items()
        }
        result["labels"] = result["input_ids"].copy()
        return result

    lm_datasets = tokenized_datasets.map(
        group_texts,
        batched=True,
        num_proc=multiprocessing.cpu_count(),
        load_from_cache_file=True,
        desc=f"Grouping texts in chunks of {seq_length}",
    )

    return lm_datasets["train"]

In [6]:
data = load_and_preprocess_data(config, MODEL_NAME, DATASET_NAME, DATASET_SUBSET, SEQ_LENGTH).select(range(500))

splits_1 = data.train_test_split(test_size=0.2, seed=RANDOM_SEED)
splits_2 = splits_1['test'].train_test_split(test_size=0.5, seed=RANDOM_SEED)


train_data = splits_1["train"]
val_data = splits_2["train"]
test_data = splits_2["test"]

print("Train dataset:")
print(train_data)

print("Val dataset:")
print(val_data)

print("Test dataset:")
print(test_data)

train_data_loader = DataLoader(
    train_data,
    batch_size=BATCH_SIZE,
    shuffle=True,
    drop_last=True,
    num_workers=1,
    prefetch_factor=2,
    collate_fn=default_data_collator
)

val_data_loader = DataLoader(
    val_data,
    batch_size=1,
    shuffle=True,
    num_workers=1,
    prefetch_factor=2,
    collate_fn=default_data_collator
)

test_data_loader = DataLoader(
    val_data,
    batch_size=1,
    shuffle=True,
    num_workers=1,
    prefetch_factor=2,
    collate_fn=default_data_collator
)

Train dataset:
Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 400
})
Val dataset:
Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 50
})
Test dataset:
Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 50
})


In [7]:
def train(model, train_data_loader, val_data_loader, test_data_loader, num_epochs, device):
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, fused=True)
    lr_scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=1000, eta_min=LR * 1e-2)

    training_times = list()
    val_times = list()
    
    for epoch in range(num_epochs):        
        train_loss = 0.0
        model.train()

        train_pbar = tqdm.notebook.tqdm(range(len(train_data_loader)), desc=f"Epoch {epoch}, train")
        start_epoch_training_time = time.time()
    
        for batch in train_data_loader:
            batch = {k: v.to(device=device) for k, v in batch.items()}
    
            outputs = model(**batch)
            
            outputs.loss.backward()
            optimizer.step()
            lr_scheduler.step()
    
            optimizer.zero_grad(set_to_none=True)
    
            train_loss += outputs.loss.item()
            train_pbar.update(1)

        training_times.append(time.time() - start_epoch_training_time)
        print(f"Train loss: {train_loss / len(train_data_loader.dataset)}")

        val_loss = 0.0
        model.eval()
        start_epoch_eval_time = time.time()
        
        with torch.no_grad():
            for batch in tqdm.notebook.tqdm(val_data_loader, desc=f"Epoch {epoch}, val"):
                batch = {k: v.to(device=device) for k, v in batch.items()}
                outputs = model(**batch)
    
                val_loss += outputs.loss.item()

        val_times.append(time.time() - start_epoch_eval_time)
        print(f"Val loss: {val_loss / len(val_data_loader.dataset)}")

    test_loss = 0.0
    model.eval()
    
    with torch.no_grad():
        for batch in tqdm.notebook.tqdm(test_data_loader, desc=f"Test"):
            batch = {k: v.to(device=device) for k, v in batch.items()}
            outputs = model(**batch)
    
            test_loss += outputs.loss.item()

    print(f"Test loss: {test_loss / len(test_data_loader.dataset)}")

    mean_training_time = np.mean(training_times)
    mean_val_time = np.mean(val_times)

    print(f"Mean training time: {mean_training_time} s")
    print(f"Mean val time: {mean_val_time} s")

    
    torch.cuda.empty_cache()

In [10]:
start_time = time.time()

train(model, train_data_loader, val_data_loader, test_data_loader, NUM_EPOCHS, device)

print(f"Full training time: {time.time() - start_time} s")

Epoch 0, train:   0%|          | 0/200 [00:00<?, ?it/s]

`loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Train loss: 3.205457183122635


Epoch 0, val:   0%|          | 0/50 [00:00<?, ?it/s]

Val loss: 5.161365356445312


...

Train loss: 2.4727014291286467


Epoch 1, val:   0%|          | 0/50 [00:00<?, ?it/s]

Val loss: 4.874359359741211


Test:   0%|          | 0/50 [00:00<?, ?it/s]

Test loss: 4.874359359741211
Mean training time: 94.00500929355621 s
Mean val time: 3.83602774143219 s
Full training time: 199.58619379997253 s


## Профилирование DL кода

Профилирование нужно для выявления проблемных мест и их последующего исправления (или нет). Профилировать можно потребление любого ресурса (времени, оперативной памяти, видео памяти и т.д.). Мы поговорим про профилирование времени (скорости) и памяти.

### Простой замер времени выполнения

Самое простое -- засечь время.

In [11]:
# Сгенерируем тестовые матрицы.
a_small = np.random.rand(16, 16).astype(np.float32)
b_small = np.random.rand(16, 16).astype(np.float32)

a_big = np.random.rand(4096, 4096).astype(np.float32)
b_big = np.random.rand(4096, 4096).astype(np.float32)

def bench_perf(a, b, num_iters: int = 100):
    times = list()

    for _ in range(num_iters):
        start_time = time.time()
        np.matmul(a, b)

        times.append(time.time() - start_time)

    mean_time = np.mean(times)

    print(f"Mean time: {mean_time} s")

print("Small matrices:")
bench_perf(a_small, b_small)
print()

print("Big matrices:")
bench_perf(a_big, b_big)

Small matrices:
Mean time: 1.8835067749023438e-06 s

Big matrices:
Mean time: 0.24890775442123414 s


А теперь попробуем тоже самое на GPU.

In [12]:
a_small_gpu = torch.from_numpy(a_small).to(device)
b_small_gpu = torch.from_numpy(b_small).to(device)

a_big_gpu = torch.from_numpy(a_big).to(device)
b_big_gpu = torch.from_numpy(b_big).to(device)

def bench_perf_gpu(a, b, num_iters: int = 100):
    times = list()

    for _ in range(num_iters):
        start_time = time.time()
        _ = torch.matmul(a, b)

        times.append(time.time() - start_time)

    mean_time = np.mean(times)

    print(f"Mean time: {mean_time} s")

print("Small matrices:")
bench_perf_gpu(a_small_gpu, b_small_gpu)
print()

print("Big matrices:")
bench_perf_gpu(a_big_gpu, b_big_gpu)
    

Small matrices:
Mean time: 7.717609405517578e-05 s

Big matrices:
Mean time: 1.2934207916259766e-05 s


Зачастую, такого способа замера производительности хватает. Но есть ньюансы работы с GPU, которые нужно учитывать.

### Учитываем ньюансы работы с GPU

CPU шедулит кернелы (операции) на GPU. Обычно CPU убегает вперёд, ставя задачи GPU. Поэтому через time.time() мы можем замерить:
  - Время постановки задач. Например, из-за того что не синхронизировались с GPU.
  - Сильно большее время. Например, когда ставим точки синхронизации в коде, из-за чего постановка кернелов в очередь для GPU постоянно прерывается.

Померить время выполнения кернелов возможно с помощью CUDA Events.

In [13]:
def bench_perf_gpu_cuda_events(a, b, num_iters: int = 100):
    start_events = [torch.cuda.Event(enable_timing=True) for _ in range(num_iters)]
    end_events = [torch.cuda.Event(enable_timing=True) for _ in range(num_iters)]

    for i in range(num_iters):
        start_events[i].record()
        torch.matmul(a, b)
        end_events[i].record()

    torch.cuda.synchronize()

    times = [start_events[i].elapsed_time(end_events[i]) for i in range(num_iters)]

    mean_time = np.mean(times)

    print("Mean time: {mean_time}")


print("Small matrices:")
bench_perf_gpu(a_small_gpu, b_small_gpu)
print()

print("Big matrices:")
bench_perf_gpu(a_big_gpu, b_big_gpu)

Small matrices:
Mean time: 0.00010017156600952148 s

Big matrices:
Mean time: 8.18490982055664e-06 s


Для более продвинутого профилирования существуют специальные программы -- профайлеры. Например, для профилирования DL кода существует Torch Profiler.

Сегодня мы эту тему не будем рассматривать.

### А что с памятью?

В основном используются 2 программы: nvidia-smi и nvtop. Они показывают загрузку GPU потребление VRAM процессами.

Давайте посмотрим, как они работают.

In [14]:
train(model, train_data_loader, val_data_loader, test_data_loader, NUM_EPOCHS, device)

Epoch 0, train:   0%|          | 0/200 [00:00<?, ?it/s]

Train loss: 2.348415707945824


Epoch 0, val:   0%|          | 0/50 [00:00<?, ?it/s]

Val loss: 4.652422938346863


Epoch 1, train:   0%|          | 0/200 [00:00<?, ?it/s]

Train loss: 2.2194956660270693


Epoch 1, val:   0%|          | 0/50 [00:00<?, ?it/s]

Val loss: 4.503679161071777


Test:   0%|          | 0/50 [00:00<?, ?it/s]

Test loss: 4.503679161071777
Mean training time: 94.18615591526031 s
Mean val time: 3.858720064163208 s


## Оптимизируем DL код

Хочется:

  - Чтобы обучение и инференс шли быстрее и требовали меньше вычислительных ресурсов.
  - Чтобы видеопамяти потреблялось меньше, для уменьшения количества необходимых для решения задачи железок.

В тренировке нейросети есть 2 основных компоненты:
  - Чтение, обработка и загрузка данных.
  - Процесс обучения весов.

С машинным временем понятно, всегда хочется, чтобы работало быстрее. А что бывает, когда не хватает памяти?

In [15]:
train_data_loader_oom = DataLoader(
    train_data,
    batch_size=int(BATCH_SIZE * 4),
    shuffle=True,
    drop_last=True,
    num_workers=1,
    prefetch_factor=2,
    collate_fn=default_data_collator
)

train(model, train_data_loader_oom, val_data_loader, test_data_loader, NUM_EPOCHS, device)

Epoch 0, train:   0%|          | 0/50 [00:00<?, ?it/s]

OutOfMemoryError: CUDA out of memory. Tried to allocate 24.00 MiB. GPU 0 has a total capacity of 7.62 GiB of which 8.69 MiB is free. Including non-PyTorch memory, this process has 7.60 GiB memory in use. Of the allocated memory 7.37 GiB is allocated by PyTorch, and 66.46 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://docs.pytorch.org/docs/stable/notes/cuda.html#optimizing-memory-usage-with-pytorch-cuda-alloc-conf)

Для начала обсудим чтение, обработку и загрузку данных.

### Чтение, обработка и загрузка данных

Для того, чтобы обучение нейросети было эффективно, важно вовремя её кормить новыми данными, иначе видеокарты будут простаивать. Поэтому, очень важно сделать чтение, подготовку и загрузку данных на видеокарты достаточно быстрыми.

Одним из самых простых способов ускорить этот процесс является их перемещение на более быстрое хранилище, например с HDD на SSD, с просто SSD на RAID массив SSD и так далее. Не редко, этого бывает достаточно.

Вторым, часто эффективным способом является кеширование. Вы выполняете как можно больше операций заранее и сохраняете результат, который будете использовать при обучении нейронки (например, нормализация картинок).

Так же, DataLoader из PyTorch имеет большое количество настроек, потюнив которые можно существенно ускорить подготовку и загрузку данных. Давайте разберём некоторые его параметры:

  - batch_size -- размер батча, чем он больше, тем больше будет потребление памяти при обучении.
  - shuffle -- перемешивать ли данные на каждой эпохе.
  - sampler -- альтернатива шафлу, определяет как извлекать данные из датасета.
  - num_workers -- количество подпроцессов для загрузки данных. Может показаться, что чем больше, тем лучше, но это не так. Увеличение количества воркеров действительно даёт ускорение загрузки данных, но лишь до определённого предела. Общие соображения по количеству воркеров такое: оно не должно быть больше количества вычислительных ядер.
  - pin_memory -- если True, data loader копирует тезоры в закреплённую память на устройстве, потом возвращает их. Опция может повысить скорость загрузки на видеокарту за счёт механизма Direct Memory Access (копирование происходит без участия CPU).
  - prefetch_factor -- количество батчей, загружаемых каждым воркером заранее. Например, при значении 2 будет загружаться заранее 2 * num_workers батчей.
  - persistent_workers -- сохранять ли воркеры после завершения прохода по датасету. Если True, воркеры в конце эпохи не завершаются и не пересоздаются в начале следующей эпохи обучения.

Давайте посмотрим, как работает batch_size:

In [8]:
train_data_loader_batch_size_1 = DataLoader(
    train_data,
    batch_size=1,
    shuffle=True,
    drop_last=True,
    num_workers=1,
    prefetch_factor=1,
    collate_fn=default_data_collator
)

train(model, train_data_loader_batch_size_1, val_data_loader, test_data_loader, NUM_EPOCHS, device)

Epoch 0, train:   0%|          | 0/400 [00:00<?, ?it/s]

`loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Train loss: 5.8289571177959445


Epoch 0, val:   0%|          | 0/50 [00:00<?, ?it/s]

Val loss: 4.951394920349121


...

Train loss: 4.803638091683387


Epoch 1, val:   0%|          | 0/50 [00:00<?, ?it/s]

Val loss: 4.81462221622467


Test:   0%|          | 0/50 [00:00<?, ?it/s]

Test loss: 4.81462221622467
Mean training time: 99.82667851448059 s
Mean val time: 3.8091843128204346 s


По сравнению со старым batch_size потребление памяти сильно уменьшилось.

### Процесс обучения весов

Есть различные подходы, которые могут помочь с задачами уменьшения потребления памяти и вычислительных ресурсов.

#### Mixed precision

Подход основан на том, что мы часть вычислений производим с пониженной точностью. Это даёт уменьшение времени счёта (если есть поддержка со стороны оборудования), снижение потребления видеопамяти на инференсе и, возможно, снижение потребления памяти при обучении.

![mixed precision scheme](images/mixed_precision.jpeg)
Источник: https://docs.fast.ai/callback.fp16.html

Mixed precision даёт ускорение вычислений, но может привести к повышенному потреблению памяти (мастер модель и градиенты хранятся в одинарной точности, мы экономим только на хранении активаций).

В mixed precision используются различные типы, например:

![Floating point types](images/fp_types.png)

Источник: https://developer.nvidia.com/blog/accelerating-ai-training-with-tf32-tensor-cores/

При использовании типа float16 требуется масштабирование, потом что диапазон значений в нём меньше, чем в типе float32. Тип bfloat16 удобен тем, что при его использовании масштабирование не нужно. Поэтому сейчас тип float16 используется только там, где нет хардварной поддержки типа bfloat16 (это крайне специфические ситуации).

Посмотрим на примере, как включить mixed precision с типом bfloat16.

In [9]:
start_time = time.time()

with torch.autocast(device_type="cuda", dtype=torch.bfloat16):
    train(model, train_data_loader, val_data_loader, test_data_loader, NUM_EPOCHS, device)

print(f"Full training time: {time.time() - start_time} s")

Epoch 0, train:   0%|          | 0/200 [00:00<?, ?it/s]

Train loss: 2.351805723309517


Epoch 0, val:   0%|          | 0/50 [00:00<?, ?it/s]

Val loss: 4.791464285850525


Epoch 1, train:   0%|          | 0/200 [00:00<?, ?it/s]

Train loss: 2.3393512135744094


Epoch 1, val:   0%|          | 0/50 [00:00<?, ?it/s]

Val loss: 4.778804898262024


Test:   0%|          | 0/50 [00:00<?, ?it/s]

Test loss: 4.778804898262024
Mean training time: 44.30066776275635 s
Mean val time: 1.460091233253479 s
Full training time: 93.03568863868713 s


Неплохое ускорение, не так ли?)

#### Gradient Accumulation

Аккумуляция градиентов была придумана для того, чтобы можно было считать с нужным размером батча, но с которым не получается поместиться в VRAM.

Идея заключается в том, что можно считать градиенты по небольшим частям батча, накопить их и после прохода всего батча сделать шаг оптимизатора.

Такой подход позволяет сэкономить на используемой памяти, но в замен увеличивается время обучения.

In [10]:
train_data_loader_grad_acc = DataLoader(
    train_data,
    batch_size=2,
    shuffle=True,
    drop_last=True,
    num_workers=1,
    prefetch_factor=2,
    collate_fn=default_data_collator
)


def train_grad_acc(model, train_data_loader, num_epochs, accumulation_steps, device):
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, fused=True)
    lr_scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=1000, eta_min=LR * 1e-2)

    training_times = list()
    val_times = list()
    
    for epoch in range(num_epochs):        
        train_loss = 0.0
        model.train()

        train_pbar = tqdm.notebook.tqdm(range(len(train_data_loader)), desc=f"Epoch {epoch}, train")
        start_epoch_training_time = time.time()
    
        for batch_num, batch in enumerate(train_data_loader):
            batch = {k: v.to(device=device) for k, v in batch.items()}
    
            outputs = model(**batch)

            loss = outputs.loss / accumulation_steps
            outputs.loss.backward()

            if ((batch_num + 1) % accumulation_steps == 0) or (batch_num + 1 == len(train_data_loader)):
                optimizer.step()
                lr_scheduler.step()
    
                optimizer.zero_grad(set_to_none=True)
    
            train_loss += loss.item()
            train_pbar.update(1)

        training_times.append(time.time() - start_epoch_training_time)
        print(f"Train loss: {train_loss / len(train_data_loader.dataset)}")

    mean_training_time = np.mean(training_times)

    print(f"Mean training time: {mean_training_time} s")

    
    torch.cuda.empty_cache()

In [11]:
start_time = time.time()

train_grad_acc(model, train_data_loader_grad_acc, NUM_EPOCHS, accumulation_steps=4, device=device)

print(f"Full training time: {time.time() - start_time} s")

Epoch 0, train:   0%|          | 0/200 [00:00<?, ?it/s]

Train loss: 0.7785344117879868


Epoch 1, train:   0%|          | 0/200 [00:00<?, ?it/s]

Train loss: 0.5807508191466332
Mean training time: 91.75054323673248 s
Full training time: 183.54682564735413 s


#### Gradient Checkpointing

Данный метод снижения потребления памяти придуман для тех ситуаций, когда мы не можем обучать нейронную сеть даже с батчем равным 1. Суть заключается в том, что мы можем сохранять не все активации для расчёта градиента при обратном проходе, а только ключевые, все остальные будут пересчитаны при расчёте градиента. Это замедляет обучение, зато снижает требования к памяти.

Применить gradient checkpointing довольно просто:

In [12]:
from torch.utils.checkpoint import checkpoint_sequential


def train_with_checkpoints(model, train_data_loader, val_data_loader, test_data_loader, num_epochs, device):
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, fused=True)
    lr_scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=1000, eta_min=LR * 1e-2)

    training_times = list()
    val_times = list()
    
    for epoch in range(num_epochs):        
        train_loss = 0.0
        model.train()

        train_pbar = tqdm.notebook.tqdm(range(len(train_data_loader)), desc=f"Epoch {epoch}, train")
        start_epoch_training_time = time.time()
    
        for batch in train_data_loader:
            batch = {k: v.to(device=device) for k, v in batch.items()}
    
            outputs = checkpoint_sequential(model, 4, batch)
            
            outputs.loss.backward()
            optimizer.step()
            lr_scheduler.step()
    
            optimizer.zero_grad(set_to_none=True)
    
            train_loss += outputs.loss.item()
            train_pbar.update(1)

        training_times.append(time.time() - start_epoch_training_time)
        print(f"Train loss: {train_loss / len(train_data_loader.dataset)}")

        val_loss = 0.0
        model.eval()
        start_epoch_eval_time = time.time()
        
        with torch.no_grad():
            for batch in tqdm.notebook.tqdm(val_data_loader, desc=f"Epoch {epoch}, val"):
                batch = {k: v.to(device=device) for k, v in batch.items()}
                outputs = model(**batch)
    
                val_loss += outputs.loss.item()

        val_times.append(time.time() - start_epoch_eval_time)
        print(f"Val loss: {val_loss / len(val_data_loader.dataset)}")

    test_loss = 0.0
    model.eval()
    
    with torch.no_grad():
        for batch in tqdm.notebook.tqdm(test_data_loader, desc=f"Test"):
            batch = {k: v.to(device=device) for k, v in batch.items()}
            outputs = model(**batch)
    
            test_loss += outputs.loss.item()

    print(f"Test loss: {test_loss / len(test_data_loader.dataset)}")

    mean_training_time = np.mean(training_times)
    mean_val_time = np.mean(val_times)

    print(f"Mean training time: {mean_training_time} s")
    print(f"Mean val time: {mean_val_time} s")

    
    torch.cuda.empty_cache()

In [13]:
start_time = time.time()

train(model, train_data_loader, val_data_loader, test_data_loader, NUM_EPOCHS, device)

print(f"Full training time: {time.time() - start_time} s")

Epoch 0, train:   0%|          | 0/200 [00:00<?, ?it/s]

Train loss: 2.19889430642128


Epoch 0, val:   0%|          | 0/50 [00:00<?, ?it/s]

Val loss: 4.456232485771179


Epoch 1, train:   0%|          | 0/200 [00:00<?, ?it/s]

Train loss: 2.095870104432106


Epoch 1, val:   0%|          | 0/50 [00:00<?, ?it/s]

Val loss: 4.34450478553772


Test:   0%|          | 0/50 [00:00<?, ?it/s]

Test loss: 4.34450478553772
Mean training time: 94.18123829364777 s
Mean val time: 3.7674591541290283 s
Full training time: 199.7120430469513 s
